# RAG biblijny – zadania

## Przygotowanie danych

### Zadanie 1 — Indeksowanie wersetów biblijnych w Qdrant

Wczytaj dane z pliku `data/polskie_przeklady.csv` oraz `data/wszystkie_ksiegi.csv`. Wybierz dowolne tłumaczenie (rekomendowana kolumna to `esp` — Edycja Świętego Pawła) i połącz je z metadanymi ksiąg. Następnie zapisz wersety do wektorowej bazy danych Qdrant w kolekcji `biblia`.

Wskazówki:
- Użyj modelu `sdadas/mmlw-retrieval-roberta-large-v2` z HuggingFace do gęstych embeddingów oraz modelu BM25 (`Qdrant/bm25`) przez `FastEmbedSparse` do wektorów rzadkich — umożliwi to późniejsze wyszukiwanie hybrydowe.
- Każdy werset powinien stanowić osobny punkt w bazie. Jako `id` punktów w kolekcji użyj kolejnych liczb całkowitych
- Payload każdego punktu powinien mieć strukturę:

```json
{
    "page_content": "Na początku Bóg stworzył niebo i ziemię.",
    "metadata": {
        "bible_part": "Stary Testament",
        "book_category": "Pięcioksiąg",
        "book_name": "Księga Rodzaju",
        "book_abbrev": "rdz",
        "deuterocanonical": false,
        "original_language": "hebrajski",
        "chapter": 1,
        "verse": 1
    }
}
```


In [1]:
import numpy as np
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest
from qdrant_client.http.models import Distance, VectorParams

2026-06-09 13:56:12.561811166 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card5": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"
2026-06-09 13:56:12.561893528 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card3": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card3/device/vendor"
2026-06-09 13:56:12.562456810 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card4": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card4/device/vendor"
2026-06-09 13:56:12.562608461 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
df_verses = (
    pd.read_csv("data/polskie_przeklady.csv", usecols=["book_abbrev", "chapter", "verse", "esp"])
    .rename(columns={"esp": "text"})
    .dropna(subset=["text"])
)

df_books = pd.read_csv("data/wszystkie_ksiegi.csv")

df = df_verses.merge(df_books, on="book_abbrev").reset_index(drop=True)
df.head()

,book_abbrev,chapter,verse,text,book_name,bible_part,book_category,deuterocanonical,original_language,n_chapters
0,rdz,1,1,Na początku Bóg stworzył niebo i ziemię.,Księga Rodzaju,Stary Testament,Pięcioksiąg,False,hebrajski,50
1,rdz,1,2,A ziemia była bezładną pustką. Ciemność zalega...,Księga Rodzaju,Stary Testament,Pięcioksiąg,False,hebrajski,50
2,rdz,1,3,"Bóg rzekł: ""Niech się stanie światłość"". I sta...",Księga Rodzaju,Stary Testament,Pięcioksiąg,False,hebrajski,50
3,rdz,1,4,"I widział Bóg, że światłość jest dobra. Oddzie...",Księga Rodzaju,Stary Testament,Pięcioksiąg,False,hebrajski,50
4,rdz,1,5,"I nazwał Bóg światłość dniem, a ciemność nocą....",Księga Rodzaju,Stary Testament,Pięcioksiąg,False,hebrajski,50


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sdadas/mmlw-retrieval-roberta-large-v2",
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
COLLECTION_NAME = "biblia"
client = QdrantClient(url="http://localhost:6333")

/home/patryk/CloudDrive/Szkolenia/Realizacje/EITT/2026-06-09_RAG_LangGraph_Agenty/.venv/lib/python3.13/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.15.5. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [7]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
        sparse_vectors_config={
            "langchain-sparse": rest.SparseVectorParams()
        },
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
)

In [8]:
import time

In [9]:
metadata_cols = ["bible_part", "book_category", "book_name", "book_abbrev", "deuterocanonical", "original_language", "chapter", "verse"]
metadatas = df[metadata_cols].to_dict("records")

start = time.perf_counter()

vector_store.add_texts(
    texts=df["text"].tolist(),
    metadatas=metadatas,
    ids=list(range(len(df))),
)
stop = time.perf_counter()
print(f"Czas: {round(stop - start, 3)}")

Czas: 545.439


## Retrieval i generacja

### Zadanie 2 — Podstawowy RAG: retrieval + generacja odpowiedzi

Napisz funkcję `rag_query`, która dla zadanego pytania wykona retrieval z bazy wektorowej i wygeneruje odpowiedź przy użyciu LLM. Funkcja powinna przyjmować:
- `question` — pytanie użytkownika
- `top_k` — liczba wersetów do pobrania
- `retrieval_mode` — tryb wyszukiwania: `"dense"`, `"sparse"` lub `"hybrid"`

Funkcja powinna zwracać słownik z kluczami `answer` (odpowiedź LLM) oraz `context` (lista słowników, gdzie każdy słownik zawiera tekst wersetu oraz wszystkie pola z payloadu, np. `book_name`, `chapter`, `verse`).

Przetestuj funkcję, zadając różne pytania z różnymi trybami wyszukiwania i wartościami `top_k`.


In [10]:
from dotenv import load_dotenv
from langchain.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

load_dotenv()

SYSTEM_PROMPT = (
    "Jesteś asystentem analizy Biblii, który pomaga użytkownikowi poznać stanowisko Biblii w określonych kwestiach. "
    "Dostaniesz w kolejnej wiadomości oryginalne pytanie użytkownika oraz wersety biblijne, które dotyczą tego pytania. "
    "Twoim zadaniem jest odpowiedzieć na pytanie użytkownika w oparciu o przytoczone fragmenty Biblii.\n\n"
    "Zasada krytyczna: odpowiadaj wyłącznie w oparciu o przytoczony kontekst. Nie używaj własnej wiedzy!!! "
    "Nie dokonuj własnej interpretacji!!! Masz być jedynie informatorem przytaczającym naukę Biblii. "
    "POD ŻADNYM POZOREM nie używaj wiedzy spoza dostarczonego tekstu.\n\n"
    "Nie musisz odwoływać się do wszystkich wersetów — użyj tylko tych, które uznasz za najlepiej pasujące do pytania. "
    "Jeżeli wszystkie fragmenty pasują, tylko wtedy możesz odnieść się do wszystkich.\n\n"
    "Twoja odpowiedź powinna być zwięzła i składać się z krótkich, trafnych akapitów przekazujących konkretną myśl. "
    "Każdy akapit powinien zawierać cytowania wersetów popierających tę myśl (jeden lub więcej). "
    "Cała odpowiedź powinna być spójna i prowadzić jedną ciągłą narrację. "
    "Cytowania mogą pochodzić WYŁĄCZNIE z przytoczonych wersetów.\n\n"
    "Cytowania (sigla) podawaj w zapisie katolickim, np. Rdz 1, 2 zamiast Rdz 1:2.\n\n"
    "Odpowiedź powinna zawierać elementy stylowania markdown (pogrubienia, bullet pointy itp.) jeżeli ma to zastosowanie. "
    'Jeśli powołujesz się na dokładny cytat tekstu Biblii, zamknij go w backtikach oraz cudzysłowiach, np. `"Ten tekst jest fragmentem Biblii"`. '
    "Nie używaj dla cytatów pogrubienia ani znaków *. "
    'Cudzysłowy powinny być dokładnie tym znakiem -> " zarówno otwierający jak i zamykający.'
)

HUMAN_PROMPT = "Pytanie użytkownika: {input}\n\nKontekst: {context}"

llm = ChatOpenAI(model="openai/gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template(HUMAN_PROMPT),
])

document_chain = create_stuff_documents_chain(llm, prompt)

In [11]:
def basic_rag_query(question: str, top_k: int, retrieval_mode: str) -> dict:
    mode_map = {
        "dense": RetrievalMode.DENSE,
        "sparse": RetrievalMode.SPARSE,
        "hybrid": RetrievalMode.HYBRID,
    }
    vs = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        sparse_embedding=sparse_embeddings,
        retrieval_mode=mode_map[retrieval_mode],
    )
    retriever = vs.as_retriever(search_kwargs={"k": top_k})
    retrieval_chain = create_retrieval_chain(retriever, document_chain)

    response = retrieval_chain.invoke({"input": question})

    return {
        "answer": response["answer"],
        "context": [
            {"verse_text": doc.page_content} | doc.metadata
            for doc in response["context"]
        ],
    }

In [12]:
result = basic_rag_query("Ile dni trwał biblijny potop?", top_k=5, retrieval_mode="hybrid")
print(result["answer"])

Na podstawie przytoczonego kontekstu, biblijny potop trwał czterdzieści dni. Fragment, który to potwierdza, mówi: `"Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią"`.


In [13]:
result

{'answer': 'Na podstawie przytoczonego kontekstu, biblijny potop trwał czterdzieści dni. Fragment, który to potwierdza, mówi: `"Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią"`.',
 'context': [{'verse_text': 'Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią.',
   'bible_part': 'Stary Testament',
   'book_category': 'Pięcioksiąg',
   'book_name': 'Księga Rodzaju',
   'book_abbrev': 'rdz',
   'deuterocanonical': False,
   'original_language': 'hebrajski',
   'chapter': 7,
   'verse': 17,
   '_id': 176,
   '_collection_name': 'biblia'},
  {'verse_text': 'I jak było za dni Noego, tak będzie w dniach Syna Człowieczego.',
   'bible_part': 'Nowy Testament',
   'book_category': 'Ewangelie',
   'book_name': 'Ewangelia Łukasza',
   'book_abbrev': 'luk',
   'deuterocanonical': False,
   'original_language': 'grecki',
   'chapter': 17,
   'verse': 26,
   '_id': 29767,
   '_collection_name': 'biblia'},
  {'verse_text': 'Ile dni zostało

Na podstawie przytoczonego kontekstu, biblijny potop trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią przez ten czas. Fragment, który to potwierdza, mówi: `"Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią."`

--- Kontekst ---
Księga Rodzaju 7,17: Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią.
Ewangelia Łukasza 17,26: I jak było za dni Noego, tak będzie w dniach Syna Człowieczego.
Księga Psalmów 119,84: Ile dni zostało Twemu słudze? Kiedy osądzisz moich prześladowców?
Księga Ezechiela 4,4: Potem połóż się na lewym boku i złóż na nim winę ludu izraelskiego. Ile dni będziesz tak leżał, tyle będziesz ponosił karę, na którą oni zasłużyli.
Księga Rodzaju 8,22: Jak długo istnieje ziemia, nie ustanie siew i żniwo, zimno i upał, lato i zima, dzień i noc".


### Zadanie 3 — Reranking pobranych wersetów

Rozszerz funkcję z poprzedniego zadania o opcjonalny reranking. Napisz nową funkcję `rag_query_with_rerank`, która przyjmuje dodatkowy parametr `rerank: bool = False`.

Do implementacji użyj `LLMListwiseRerank` (jako `base_compressor`) owinięty w `ContextualCompressionRetriever`. Gdy `rerank=False`, przekaż do łańcucha zwykły retriever bez kompresji.

Porównaj odpowiedzi i pobrane dokumenty z rerankingiem i bez — zwróć uwagę czy kolejność i dobór wersetów się zmienia.

In [14]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

In [ ]:
def rag_query_with_rerank(question: str, top_k: int, retrieval_mode: str, rerank: bool = False) -> dict:
    mode_map = {
        "dense": RetrievalMode.DENSE,
        "sparse": RetrievalMode.SPARSE,
        "hybrid": RetrievalMode.HYBRID,
    }
    vs = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        sparse_embedding=sparse_embeddings,
        retrieval_mode=mode_map[retrieval_mode],
    )
    base_retriever = vs.as_retriever(search_kwargs={"k": top_k})

    if rerank:
        reranker = LLMListwiseRerank.from_llm(llm=llm, top_n=top_k)
        retriever = ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=base_retriever,
        )
    else:
        retriever = base_retriever

    retrieval_chain = create_retrieval_chain(retriever, document_chain)
    response = retrieval_chain.invoke({"input": question})

    return {
        "answer": response["answer"],
        "context": [
            {"verse_text": doc.page_content} | doc.metadata
            for doc in response["context"]
        ],
    }


result = rag_query_with_rerank("Ile dni trwał biblijny potop?", top_k=5, retrieval_mode="hybrid", rerank=True)
print(result["answer"])
print("\n--- Kontekst ---")
for v in result["context"]:
    print(f"{v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text']}")

In [ ]:
result = rag_query_with_rerank("Ile dni trwał biblijny potop?", top_k=5, retrieval_mode="hybrid", rerank=False)
print(result["answer"])
print("\n--- Kontekst ---")
for v in result["context"]:
    print(f"{v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text']}")

### Zadanie 4 — Filtrowanie payloadu i opcjonalna generacja

Napisz funkcję `rag_query_with_filters`, która rozszerza poprzednie rozwiązanie o dwie nowe możliwości:

**Filtrowanie payloadu** — funkcja powinna przyjmować opcjonalny parametr `filters` z obiektem `models.Filter` z biblioteki `qdrant_client`. Filtry pozwalają zawęzić wyszukiwanie wyłącznie do wersetów spełniających określone warunki metadanych (np. tylko Stary Testament, bez ksiąg deuterokanonicznych, tylko konkretna kategoria ksiąg). Ponieważ filtry muszą być przekazane dynamicznie w momencie wywołania, użyj `configurable_fields` na retrieverze wraz z `ConfigurableField` — dzięki temu `search_kwargs` (w tym `filter`) można podać w `config` przy każdym wywołaniu zamiast przy tworzeniu retrievera.

**Opcjonalne generowanie odpowiedzi** — dodaj parametr `generate_answer: bool = True`. Gdy `True`, uruchom pełny łańcuch RAG i zwróć odpowiedź LLM. Gdy `False`, wywołaj retriever bezpośrednio z pytaniem i zwróć tylko pobrany kontekst bez angażowania LLM.

Funkcja powinna zwracać słownik z `answer` (lub `None` gdy `generate_answer=False`) oraz `context` z pełnymi metadanymi każdego wersetu.

Napisz też pomocniczą funkcję `show_context`, która w czytelny sposób wydrukuje listę wersetów z kontekstu (m.in. część Biblii, kategorię księgi, nazwę, rozdział i werset).

Przetestuj różne typy filtrów: `MatchValue`, `MatchAny`, `MatchText`, kombinacje `must`/`should`/`must_not` oraz reranking z filtrami.


In [ ]:
# ...

## Asystent obsługujący retrieval

### Zadanie 5 — Asystent generujący zapytania do retrievera

Stwórz funkcję `generate_retrieval_requests`, która pełni rolę inteligentnego asystenta przygotowującego zapytania do bazy wektorowej. Funkcja przyjmuje:
- `question` – oryginalne pytanie użytkownika
- `analysis_level` – poziom analizy: `"basic"`, `"medium"` lub `"deep"`

Asystent powinien wygenerować 2–3 zapytania (lub jedno, jeśli pytanie jest bardzo specyficzne), z których każde pokrywa inny aspekt tematu — dzięki temu retrieval znajdzie szerszy zestaw trafnych wersetów. 

Przykład:
- Pytanie użytkownika: "Nakazy od Boga względem człowieka"
- Wygenerowane zapytanie 1: `query_text` - "Przykazania od Boga wobec Izraelitów", `query_filters` - `<tylko stary testament>`, `top_k` - 20
- Wygenerowane zapytanie 2: `query_text` - "Przykazania Jezusa wobec jego wyznawców", `query_filters` - `<tylko nowy testament>`, `top_k` - 20

Każde zapytanie to słownik z kluczami:
- `query_text` — przeformułowane pytanie zoptymalizowane pod wyszukiwanie semantyczne (bez nazw ksiąg/części Biblii, które trafią do filtrów)
- `query_filters` — gotowy obiekt `models.Filter` z `qdrant_client` lub `None`, gdy filtrowanie nie jest potrzebne; filtry stosuj **wyłącznie** gdy użytkownik wprost wskazuje konkretną część Biblii
- `top_k` — liczba wyników dobrana do poziomu analizy i popularności tematu

Do wygenerowania zapytań użyj LLM ze structured output (Pydantic + `with_structured_output`). Zdefiniuj modele Pydantic opisujące strukturę pojedynczego zapytania i całej listy.

Aby asystent wiedział jakie wartości są prawidłowe dla filtrów, wczytaj plik `data/unique_metadata_values.json` i użyj zawartych w nim informacji.

Ustal samodzielnie mapowanie pomiędzy `analysis_level` → zakres `top_k` . Im bardziej dogłębna analiza, tym więcej wersetów nalezy wyciągnąć.

In [ ]:
# ...

### Zadanie 6 — Rozszerzanie kontekstu wersetu o sąsiednie wersety

Napisz funkcję `expand_verse_context`, która dla podanego wersetu pobiera z bazy danych sąsiednie wersety z tego samego rozdziału i zwraca rozszerzony kontekst.

Ponieważ wersety zostały zaindeksowane z sekwencyjnymi identyfikatorami całkowitymi (0, 1, 2, …), sąsiedni werset poprzedzający ma `id - 1`, a następny `id + 1`. Użyj metody `client.retrieve()` do pobrania tych punktów bezpośrednio po ID.

Uwaga: id±1 może należeć do innego rozdziału lub nawet innej księgi (na granicy rozdziałów). Po pobraniu punktów odfiltruj te, których numer rozdziału (`metadata.chapter`) różni się od rozdziału wersetu wejściowego.

Funkcja powinna zwracać słownik będący połączeniem oryginalnego wersetu (słownika) oraz:
- `context_text` — połączony tekst sąsiednich wersetów (poprzedni + oryginalny + następny, o ile są w tym samym rozdziale)
- `context_verse_start` — numer pierwszego wersetu w oknie kontekstu
- `context_verse_stop` — numer ostatniego wersetu w oknie kontekstu

Przykłady:
- `Rdz 2,4` → kontekst obejmuje `Rdz 2, 3-5`
- `Rdz 2,1` → kontekst obejmuje `Rdz 2, 1-2` (brak poprzedniego wersetu w tym rozdziale)


In [ ]:
# ...

### Zadanie 7 — Filtrowanie nietrafionych wersetów przez LLM

Napisz funkcję `filter_irrelevant_verses`, która dla listy wersetów (z rozszerzonym kontekstem z ćwiczenia 6) oceni przy użyciu LLM, czy każdy z nich rzeczywiście odpowiada na pytanie użytkownika.

Funkcja powinna przyjmować:
- `user_query` — oryginalne pytanie użytkownika
- `verses` — lista słowników z rozszerzonym kontekstem (pole `context_text` z ćwiczenia 6)

Dla każdego wersetu wyślij do LLM pytanie użytkownika oraz `context_text` (a nie sam werset) — szersze okno kontekstu pozwala LLM lepiej ocenić trafność. Użyj structured output z modelem Pydantic zawierającym pole `is_relevant: bool` oraz uzasadnienie decyzji.

Ponieważ wersetów może być wiele, wywołaj LLM dla wszystkich naraz przy użyciu `chain.batch()` z parametrem `max_concurrency` — to znacznie przyspiesza przetwarzanie w porównaniu do sekwencyjnych wywołań.

Funkcja powinna zwracać słownik z kluczami:
- `relevant_verses` — lista trafnych wersetów, każdy wzbogacony o pole z uzasadnieniem trafności
- `irrelevant_verses` — lista odfiltrowanych wersetów, każdy wzbogacony o pole z uzasadnieniem odrzucenia


In [ ]:
# ...

### Zadanie 8 — Grupowanie wersetów w kategorie tematyczne

Napisz funkcję `group_verses`, która pogrupuje trafne wersety w kategorie tematyczne odpowiadające różnym aspektom odpowiedzi na pytanie użytkownika.

Funkcja przyjmuje:
- `user_query` — oryginalne pytanie użytkownika
- `relevant_verses` — lista trafnych wersetów z zadania 7

Przygotuj dla LLM wejście w postaci listy słowników `{"id": ..., "text": ...}` (gdzie `text` to `context_text`) i przekaż je jako wiadomość ludzką w JSON. W wiadomości systemowej poinstruuj LLM, żeby podzielił wszystkie cytaty na niezależne tematycznie grupy i nazwał je tak, by nadawały się jako nagłówki sekcji odpowiedzi. Każdy werset musi trafić do przynajmniej jednej kategorii; jeśli pasuje do kilku — może pojawić się w wielu.

Użyj structured output — zdefiniuj modele Pydantic. Następnie za pomocą słownika `{id: verse}` rozwiąż ID z powrotem na pełne słowniki wersetów.

Funkcja powinna zwracać słownik `{nazwa_kategorii: [verse_dict, ...]}`.

Aby zobaczyć jak powinien wyglądać efekt grupowania w praktyce, odwiedź stronę [biblia.palej.dev](https://biblia.palej.dev/) i przejrzyj przykładowe analizy — nazwy aspektów to nagłówki poszczególnych sekcji odpowiedzi.


In [ ]:
# ...

### Zadanie 9 — Pobieranie pełnych tekstów rozdziałów

Napisz funkcję `retrieve_chapters`, która dla każdej kategorii z wyników zadania 8 pobierze pełne teksty rozdziałów, do których należą pogrupowane wersety.

Funkcja przyjmuje słownik `grouped_verses` (wynik `group_verses`) i zwraca słownik o tej samej strukturze kluczy, ale zamiast wersetów — listę rozdziałów: `{nazwa_kategorii: [chapter_dict, ...]}`.

Każdy `chapter_dict` powinien zawierać: identyfikator rozdziału (np. `"Księga Rodzaju 1"`), nazwę księgi, numer rozdziału oraz pełny tekst rozdziału złożony ze wszystkich jego wersetów.

Wskazówki implementacyjne:
- Do pobrania wszystkich wersetów danego rozdziału użyj `client.scroll()` z filtrem na `book_name` i `chapter` — zwróci wszystkie punkty spełniające warunek bez limitu wektorowego.
- Złóż tekst rozdziału łącząc kolejne wersety w formacie `numer_wersetu: treść_wersetu` oddzielone znakiem nowej linii.
- Zaimplementuj cache słownikowy indeksowany kluczem `"Nazwa księgi numer_rozdziału"` — ten sam rozdział może być potrzebny dla kilku kategorii, a cache zapobiegnie wielokrotnym zapytaniom do bazy.


In [ ]:
# ...

### Zadanie 10 — Generowanie odpowiedzi per aspekt

Napisz funkcję `generate_answers_per_aspect`, która wygeneruje ustrukturyzowaną odpowiedź dla każdej kategorii tematycznej z ćwiczenia 8.

Funkcja przyjmuje:
- `user_query` — oryginalne pytanie użytkownika
- `grouped_verses` — wynik `group_verses` (słownik kategorii z wersetami)
- `retrieved_chapters` — wynik `retrieve_chapters` (słownik kategorii z rozdziałami)

Dla każdego aspektu/kategorii zbuduj kontekst łączący dwa poziomy szczegółowości: najpierw wylistuj konkretne trafne wersety z danego rozdziału (numer i treść), a potem dołącz pełny tekst rozdziału — dzięki temu LLM ma zarówno celne fragmenty, jak i szerszy kontekst narracyjny.

Użyj structured output z hierarchicznym modelem Pydantic:
- `BibleCitation` — nazwa księgi, numer rozdziału, numer wersetu
- `ResponseParagraph` — tekst akapitu + lista cytowań (`list[BibleCitation]`)
- `AspectResponse` — nazwa aspektu + lista akapitów (`list[ResponseParagraph]`)

Wszystkie aspekty przetwarzaj równolegle przy użyciu `chain.batch()` z `max_concurrency`.

Funkcja powinna zwracać listę słowników (`.model_dump()` na każdym `AspectResponse`) — jednego na każdą kategorię.


In [ ]:
# ...

## Pipeline w langgraph

### Zadanie 11 — Pipeline RAG w LangGraph

Przekształć funkcje z zadań 5–10 w spójny pipeline oparty na LangGraph. Celem jest zamknięcie całego przepływu — od pytania użytkownika aż po listę wygenerowanych odpowiedzi — w jeden graf, który można wywołać jedną instrukcją.

**Struktura grafu**

Graf jest liniowy (brak rozgałęzień warunkowych) i składa się z siedmiu węzłów wykonywanych sekwencyjnie:

```
START
  └─► generate_requests      # zadanie 5: generuje listę zapytań do retrievera
  └─► execute_retrieval      # zadanie 2/4: pobiera wersety z Qdrant
  └─► expand_neighbors       # zadanie 6: rozszerza kontekst o sąsiednie wersety
  └─► filter_irrelevant      # zadanie 7: filtruje nieistotne wersety przez LLM
  └─► group_results          # zadanie 8: grupuje trafne wersety w kategorie
  └─► retrieve_chapters      # zadanie 9: pobiera pełne teksty rozdziałów
  └─► generate_answer        # zadanie 10: generuje odpowiedź per aspekt
END
```

**Stan grafu**

Zdefiniuj klasę stanu jako `TypedDict`. Powinna zawierać:
- `user_query` — wejściowe pytanie użytkownika
- po jednym polu na dane wyjściowe każdego węzła (np. `retrieval_requests`, `retrieved_verses`, `expanded_verses`, `relevant_verses`, `grouped_verses`, `retrieved_chapters`, `answers`)